In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
df_test = pd.read_csv('/kaggle/input/datasets/ziya07/iot-edge-processed-traffic-and-incident-dataset/iot_edge_computing_public_management.csv', nrows=1)
print(df_test.columns.tolist())

In [ ]:
%%writefile traffic_engine.cpp
#include <iostream>
#include <vector>
#include <cmath>
#include <fstream>
#include <sstream>
#include <map>
#include <algorithm>

struct SensorStats {
    std::vector<double> speed_history;
};

int main() {
    std::ifstream file("/kaggle/input/datasets/ziya07/iot-edge-processed-traffic-and-incident-dataset/iot_edge_computing_public_management.csv");
    std::string line;
    std::map<std::string, SensorStats> storage;
    std::getline(file, line); 

    while (std::getline(file, line)) {
        std::stringstream ss(line);
        std::string val;
        std::vector<std::string> row;
        while (std::getline(ss, val, ',')) row.push_back(val);
        if (row.size() < 3) continue;
        storage[row[1]].speed_history.push_back(std::stod(row[2]));
    }

    std::ofstream output("bi_foundation.csv");
    output << "JoinKey,Avg_Speed,P95_Speed,Std_Dev\n";

    for (auto const& [id, stats] : storage) {
        std::vector<double> s = stats.speed_history;
        std::sort(s.begin(), s.end());
        double sum = 0, sq_sum = 0;
        for(double v : s) { sum += v; sq_sum += v*v; }
        double avg = sum / s.size();
        double stdev = sqrt((sq_sum / s.size()) - (avg * avg));
        double p95 = s[static_cast<int>(s.size() * 0.95)];
        output << id << "," << avg << "," << p95 << "," << stdev << "\n";
    }
    return 0;
}

In [ ]:
!g++ -O3 traffic_engine.cpp -o engine && ./engine

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('/kaggle/input/datasets/ziya07/iot-edge-processed-traffic-and-incident-dataset/iot_edge_computing_public_management.csv')
df_stats = pd.read_csv('bi_foundation.csv')

# Merge and Calculate BI/PTI
df = pd.merge(df_raw, df_stats, left_on='sensor_id', right_on='JoinKey')
df['Buffer_Index'] = (df['P95_Speed'] - df['Avg_Speed']) / df['Avg_Speed']
df['Planning_Time_Index'] = df['P95_Speed'] / 30.0 # Assuming 30km/h as threshold
df.to_csv('refined_features.csv', index=False)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os

# Suppress background logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# 1. Sequence Preparation
# Ensure 'df' from Module 2 is ready
data = df['Buffer_Index'].values.reshape(-1, 1)
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

def create_sequences(data, window=10):
    x, y = [], []
    for i in range(len(data)-window):
        x.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(x), np.array(y)

X_lstm, y_lstm = create_sequences(scaled_data)

# 2. Modern Sequential Model (Fixed Warning)
model = Sequential([
    layers.Input(shape=(10, 1)),        # Explicit Input Layer
    layers.LSTM(64, return_sequences=True, activation='tanh'),
    layers.Dropout(0.2),                # Prevents Overfitting
    layers.LSTM(32, activation='tanh'),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# 3. Training
print("Starting LSTM Training...")
model.fit(X_lstm, y_lstm, epochs=10, batch_size=32, verbose=0)
print("LSTM Training Complete. Future BI Prediction Model is Ready.")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# 1. Generate LSTM Predictions for the entire dataset
# We use the scaler and model from Module 3 to create a 'Predicted_Reliability' column
X_all, _ = create_sequences(scaled_data)
predictions = model.predict(X_all)
# Pad the beginning with the mean to match the original dataframe length
full_preds = np.append(np.repeat(np.nan, 10), scaler.inverse_transform(predictions))
df['LSTM_Future_Risk'] = full_preds

# 2. Create the Visual Map
fig = px.scatter_mapbox(df.dropna(subset=['LSTM_Future_Risk']), 
                        lat="latitude", 
                        lon="longitude", 
                        color="LSTM_Future_Risk", 
                        size="vehicle_speed (km/h)",
                        color_continuous_scale='Portland', # High-contrast professional scale
                        hover_name="sensor_id",
                        hover_data=["traffic_pattern", "Buffer_Index", "LSTM_Future_Risk"],
                        mapbox_style="carto-positron", # Clean, light, and attractive background
                        zoom=11,
                        height=300,
                        title="<b>LSTM Traffic Reliability Forecast</b><br><sup>Size = Current Speed | Color = Predicted Future Congestion Risk</sup>")

# 3. Aesthetics & UI Adjustments
fig.update_layout(
    margin={"r":0,"t":50,"l":0,"b":0},
    coloraxis_colorbar=dict(
        title="Predictive Risk",
        thicknessmode="pixels", thickness=20,
        lenmode="pixels", len=100,
        yanchor="top", y=1,
        ticks="outside"
    )
)

fig.show()

In [ ]:
from IPython.display import HTML
import json

# 1. Prepare a sample forecast using the LSTM for the UI
# We'll take the last window of data and project it forward 24 steps
last_window = scaled_data[-10:].reshape(1, 10, 1)
forecast = []
current_window = last_window

for _ in range(24):
    pred = model.predict(current_window, verbose=0)
    forecast.append(float(scaler.inverse_transform(pred)[0][0]))
    # Roll the window: remove first, append the new prediction
    current_window = np.append(current_window[:, 1:, :], pred.reshape(1, 1, 1), axis=1)

# 2. Encode forecast for JavaScript
forecast_json = json.dumps(forecast)

# 3. Enhanced Interactive Dashboard
js_dashboard = f"""
<div style="background:#f8f9fa; color:#333; padding:20px; border-radius:12px; border:1px solid #dee2e6; font-family:sans-serif; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
    <h3 style="color:#007bff; margin-top:0;">🔮 Proactive Reliability Forecast</h3>
    <p style="font-size:0.9em; color:#666;">This terminal uses the <b>LSTM Neural Network</b> to predict the Buffer Index for the next 24 hours.</p>
    
    <div style="display:flex; gap:20px; align-items:center; margin-bottom:20px;">
        <div style="flex:1;">
            <label style="font-weight:bold; font-size:0.8em;">SELECT SENSOR ID:</label>
            <select id="sensor-select" style="width:100%; padding:8px; border-radius:5px; border:1px solid #ccc;">
                <option>Sensor_Alpha_01</option>
                <option>Sensor_Beta_02</option>
                <option>Sensor_Gamma_03</option>
            </select>
        </div>
        <div style="flex:1;">
            <button onclick="animateForecast()" style="width:100%; background:#007bff; color:white; border:none; padding:10px; border-radius:5px; cursor:pointer; font-weight:bold;">GENERATE 24H TREND</button>
        </div>
    </div>

    <canvas id="forecastChart" width="800" height="300"></canvas>
</div>

<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
    const forecastData = {forecast_json};
    let chart;

    function animateForecast() {{
        const ctx = document.getElementById('forecastChart').getContext('2d');
        
        if (chart) chart.destroy();
        
        chart = new Chart(ctx, {{
            type: 'line',
            data: {{
                labels: Array.from({{length: 24}}, (_, i) => (i+1) + "h"),
                datasets: [{{
                    label: 'Predicted Buffer Index (Unreliability)',
                    data: forecastData,
                    borderColor: '#007bff',
                    backgroundColor: 'rgba(0, 123, 255, 0.1)',
                    fill: true,
                    tension: 0.4,
                    pointRadius: 4
                }}]
            }},
            options: {{
                responsive: true,
                plugins: {{ legend: {{ display: true }} }},
                scales: {{
                    y: {{ beginAtZero: false, title: {{ display: true, text: 'BI Value' }} }},
                    x: {{ title: {{ display: true, text: 'Hours from Now' }} }}
                }}
            }}
        }});
    }}
    
    // Initial call
    window.onload = animateForecast;
</script>
"""
display(HTML(js_dashboard))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
sns.regplot(data=df, x='Avg_Speed', y='Buffer_Index', scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title("The Reliability Gap: Average Speed vs. Buffer Index")
plt.show()

In [ ]:
%%writefile stress_test.cpp
#include <iostream>
#include <vector>
#include <random>
#include <fstream>

int main() {
    std::default_random_engine generator;
    // Normal distribution of speed centered at 45km/h
    std::normal_distribution<double> distribution(45.0, 10.0); 
    
    std::ofstream output("stress_results.csv");
    output << "Iteration,Simulated_Speed,Induced_BI\n";

    for (int i = 0; i < 10000; ++i) {
        double speed = distribution(generator);
        if (speed < 5) speed = 5; // Floor speed
        
        // Induced BI calculation: how much variance is added by this 'shock'
        double induced_bi = std::abs(60.0 - speed) / 60.0;
        output << i << "," << speed << "," << induced_bi << "\n";
    }
    return 0;
}

In [ ]:
!g++ -O3 stress_test.cpp -o stress && ./stress

In [ ]:
# Calculating the 'Cost of Unreliability'
# Formula: Cost = (Buffer Time) * (Hourly Wage) * (Number of Vehicles)
hourly_wage = 25.0  # Assumed average wage in USD
df['Buffer_Time_Min'] = (df['Buffer_Index'] * 30) # Assuming a 30-min base trip
df['Daily_Economic_Loss'] = df['Buffer_Time_Min'] * (hourly_wage / 60) * df['vehicle_speed (km/h)'] # Simplified proxy for volume

print(f"Total Daily Economic Loss due to Unreliability: ${df['Daily_Economic_Loss'].sum():,.2f}")

# Visualization of Loss per Sensor
import seaborn as sns
plt.figure(figsize=(10, 5))
sns.histplot(df['Daily_Economic_Loss'], kde=True, color='gold')
plt.title("Distribution of Daily Financial Loss per Sensor Location")
plt.show()

In [ ]:
import numpy as np

# 1. Prepare Data: Aggregate to ensure unique speed values
# We group by speed and take the mean PTI to avoid 'divide by zero' on duplicate speeds
sensitivity_df = df.groupby('vehicle_speed (km/h)')['Planning_Time_Index'].mean().reset_index()
sensitivity_df = sensitivity_df.sort_values(by='vehicle_speed (km/h)')

# 2. Calculate Gradient with handling for small intervals
speeds = sensitivity_df['vehicle_speed (km/h)'].values
pti_values = sensitivity_df['Planning_Time_Index'].values

# Calculate gradient: d(PTI) / d(Speed)
# We use a try-except or check for empty arrays to ensure stability
if len(speeds) > 1:
    sensitivity_df['Reliability_Sensitivity'] = np.gradient(pti_values, speeds)
else:
    sensitivity_df['Reliability_Sensitivity'] = 0

# 3. Clean up any remaining NaN or Inf values
sensitivity_df.replace([np.inf, -np.inf], np.nan, inplace=True)
sensitivity_df.fillna(0, inplace=True)

# 4. High-Functioning Visualization
fig = px.line(sensitivity_df, x='vehicle_speed (km/h)', y='Reliability_Sensitivity', 
              template="plotly_white",
              title="<b>Reliability Sensitivity Analysis</b><br><sup>Rate of Change in Planning Time per 1 km/h Speed Variance</sup>")

fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Stability Threshold")
fig.show()

In [ ]:
from IPython.display import HTML
current_bi_avg = df['Buffer_Index'].mean()

js_cost_calc = f"""
<div style="background:#fff3cd; color:#856404; padding:20px; border:1px solid #ffeeba; border-radius:10px;">
    <h4>💰 Commuter Loss Calculator</h4>
    <p>Based on current <b>BI: {current_bi_avg:.2f}</b></p>
    <label>Trip Distance (km): </label><input type="number" id="dist" value="15" style="width:50px;"><br>
    <label>Your Hourly Wage ($): </label><input type="number" id="wage" value="20" style="width:50px;"><br>
    <button onclick="calcLoss()" style="margin-top:10px;">Calculate My Monthly Loss</button>
    <h3 id="loss-result" style="color:#721c24;">$0.00</h3>
</div>

<script>
    function calcLoss() {{
        const d = document.getElementById('dist').value;
        const w = document.getElementById('wage').value;
        const bi = {current_bi_avg};
        // Calculation: (Time lost to BI) * Wage * 22 working days
        const timeLostHours = (d / 40) * bi; 
        const monthlyLoss = (timeLostHours * w * 22).toFixed(2);
        document.getElementById('loss-result').innerText = "$" + monthlyLoss + " / month";
    }}
</script>
"""
display(HTML(js_cost_calc))